<a href="https://colab.research.google.com/github/sara-iqbal/Alternatives_fund_lifecycle/blob/main/alternatives_fund_lifecycle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alternatives Fund Investor Lifecycle Tracker
### GAAPS Alternatives

**Author:** Sara Iqbal | MSc Data Science

---

## What This Project Does

Simulates the complete investor lifecycle for a $500M alternatives fund from initial onboarding through to fund liquidation:

```
Investor Onboarding & AML/KYC
        |
Subscription Document Review
        |
Compliance Sign-off (MLRO/FATCA/CRS)
        |
Capital Calls & Distributions (12 months)
        |
Reporting Schedule (Monthly/Quarterly/Annual)
        |
Side Letter Obligation Tracking
        |
Fund Liquidation Checklist
        |
Excel Export (Investor Register, Cap Accounts, Distribution Notices)
```

**Key metrics:**
- 50 investors across 8 jurisdictions onboarded with AML/KYC status tracking
- 12-month capital call and distribution schedule across a $500M fund
- 600+ annual reports scheduled and tracked across all investors
- Side letter obligations tracked per investor with deadline monitoring
- 24-item fund liquidation checklist with automated status tracking

In [1]:
# Step 1 — Install dependencies
!pip install pandas numpy openpyxl -q
print('Done')

Done


In [2]:
# Step 2 — Imports
import numpy as np
import pandas as pd
from datetime import date, timedelta
import json, warnings, os
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded')

Libraries loaded


In [3]:
# Step 3 — Fund Universe & Configuration

FUND_TYPES = ['Private Equity', 'Private Credit', 'Real Assets', 'Hedge Fund']

FUND_CONFIG = {
    'BLK-PE-FLAGSHIP': {'name': 'BlackRock Private Equity Flagship Fund VI', 'type': 'Private Equity',  'target_size_m': 500},
}

JURISDICTIONS = ['United States', 'United Kingdom', 'European Union', 'Switzerland',
                 'Singapore', 'Japan', 'UAE', 'Canada']

# AML/KYC risk rating thresholds by jurisdiction
JX_RISK_PROFILE = {
    'United States':   'Low',
    'United Kingdom':  'Low',
    'European Union':  'Low',
    'Switzerland':     'Medium',
    'Singapore':       'Medium',
    'Canada':          'Low',
    'Japan':           'Low',
    'UAE':             'High',
}

# FATCA/CRS classification by jurisdiction
JX_FATCA = {
    'United States':   'FATCA W-9',
    'United Kingdom':  'CRS',
    'European Union':  'CRS',
    'Switzerland':     'CRS + FATCA W-8BEN',
    'Singapore':       'CRS',
    'Japan':           'CRS',
    'UAE':             'FATCA W-8BEN',
    'Canada':          'CRS',
}

# Reporting frequencies by fund type
REPORTING_FREQ = {
    'Private Equity':  'Quarterly',
    'Private Credit':  'Quarterly',
    'Real Assets':     'Quarterly',
    'Hedge Fund':      'Monthly',
}

INVESTOR_TYPES = ['Pension Fund', 'Sovereign Wealth Fund', 'Endowment',
                  'Family Office', 'Insurance Co', 'Fund of Funds']

REPORT_TYPES = ['NAV Statement', 'Capital Account Statement', 'Quarterly Report',
                'Annual Report', 'ESG Report', 'ILPA Report']

print(f'Fund: {list(FUND_CONFIG.values())[0]["name"]}')
print(f'Target size: ${list(FUND_CONFIG.values())[0]["target_size_m"]}M')
print(f'Jurisdictions covered: {len(JURISDICTIONS)}')
print(f'Fund strategies: {len(FUND_TYPES)}')

Fund: BlackRock Private Equity Flagship Fund VI
Target size: $500M
Jurisdictions covered: 8
Fund strategies: 4


In [4]:
# Step 4 — Investor Universe Generation
# 50 investors across 8 jurisdictions with realistic AML/KYC profiles

np.random.seed(42)
investors = []

for i in range(50):
    jx    = np.random.choice(JURISDICTIONS)
    itype = np.random.choice(INVESTOR_TYPES)
    fund  = np.random.choice(FUND_TYPES)
    comm  = round(np.random.uniform(2, 20), 1)

    # KYC status distribution: 76% approved, 16% pending, 8% escalated
    kyc_roll = np.random.random()
    if kyc_roll < 0.76:
        kyc = 'Approved'
    elif kyc_roll < 0.92:
        kyc = 'Pending'
    else:
        kyc = 'Escalated'

    # Risk rating — escalated investors always high risk
    base_risk = JX_RISK_PROFILE[jx]
    if kyc == 'Escalated':
        risk = 'High'
    elif base_risk == 'High' or np.random.random() < 0.2:
        risk = 'Medium'
    else:
        risk = base_risk

    # Subscription document status
    sub_doc = 'Complete' if kyc == 'Approved' else ('Under Review' if kyc == 'Pending' else 'Incomplete')

    # Side letter — 46% of approved investors have bespoke terms
    side_letter = (kyc == 'Approved') and (np.random.random() < 0.46)

    # Side letter obligations
    sl_obligations = []
    if side_letter:
        possible = ['Monthly NAV', 'Quarterly ESG Report', 'Fee Cap 1.5%',
                    'ILPA Reporting', 'Most Favoured Nation', 'Co-investment Rights',
                    'Excuse Rights', 'Audit Inspection Rights']
        n = np.random.randint(1, 4)
        sl_obligations = list(np.random.choice(possible, n, replace=False))

    # Capital called (only for approved investors)
    called = round(comm * np.random.uniform(0.3, 0.7), 2) if kyc == 'Approved' else 0.0

    # MLRO signoff
    mlro = 'Signed off' if kyc == 'Approved' else ('Pending review' if kyc == 'Pending' else 'Escalated to MLRO')

    investors.append({
        'investor_id':      f'LP-{str(i+1).zfill(3)}',
        'investor_name':    f'{itype} {jx.split()[0]} {i+1}',
        'investor_type':    itype,
        'jurisdiction':     jx,
        'fund_type':        fund,
        'commitment_m':     comm,
        'kyc_status':       kyc,
        'risk_rating':      risk,
        'fatca_crs':        JX_FATCA[jx],
        'sub_doc_status':   sub_doc,
        'side_letter':      side_letter,
        'sl_obligations':   ', '.join(sl_obligations) if sl_obligations else 'None',
        'mlro_signoff':     mlro,
        'capital_called_m': called,
        'uncalled_comm_m':  round(comm - called, 2),
        'reporting_freq':   REPORTING_FREQ[fund],
        'onboarding_date':  f'2024-{str(np.random.randint(1,6)).zfill(2)}-{str(np.random.randint(1,28)).zfill(2)}',
    })

inv_df = pd.DataFrame(investors)
total_comm = inv_df['commitment_m'].sum()

print(f'Investors generated: {len(inv_df)}')
print(f'Total commitments: ${total_comm:.1f}M')
print(f'Approved: {len(inv_df[inv_df.kyc_status=="Approved"])}')
print(f'Pending:  {len(inv_df[inv_df.kyc_status=="Pending"])}')
print(f'Escalated:{len(inv_df[inv_df.kyc_status=="Escalated"])}')
print(f'Side letters: {inv_df.side_letter.sum()}')
print(f'\nJurisdiction breakdown:')
print(inv_df.groupby("jurisdiction")["investor_id"].count().to_string())

Investors generated: 50
Total commitments: $550.8M
Approved: 39
Pending:  5
Escalated:6
Side letters: 20

Jurisdiction breakdown:
jurisdiction
Canada            3
European Union    3
Japan             8
Singapore         9
Switzerland       8
UAE               6
United Kingdom    6
United States     7


In [5]:
# Step 5 — Capital Call & Distribution Engine
# 12-month simulation of capital calls and distributions

np.random.seed(77)
MONTHS = pd.date_range('2024-01-01', periods=12, freq='MS')

# Fund-level monthly activity
call_amounts    = np.round(np.random.uniform(5, 33, 12), 1)
dist_amounts    = np.round(np.random.uniform(3, 21, 12), 1)
call_purposes   = np.random.choice(
    ['Portfolio Investment','Management Fees','Fund Expenses','Follow-on Investment'], 12)

approved_inv = inv_df[inv_df['kyc_status'] == 'Approved'].copy()
approved_comm_total = approved_inv['commitment_m'].sum()

call_records = []
dist_records = []

for m_idx, month in enumerate(MONTHS):
    call_amt = call_amounts[m_idx]
    dist_amt = dist_amounts[m_idx]

    for _, inv in approved_inv.iterrows():
        # Pro-rata allocation based on commitment
        pct = inv['commitment_m'] / approved_comm_total

        # Capital call notice
        inv_call = round(call_amt * pct * 1_000_000, 2)
        call_records.append({
            'date':          month.strftime('%Y-%m-%d'),
            'investor_id':   inv['investor_id'],
            'investor_name': inv['investor_name'],
            'jurisdiction':  inv['jurisdiction'],
            'call_type':     'Capital Call',
            'purpose':       call_purposes[m_idx],
            'fund_total_m':  call_amt,
            'investor_pct':  round(pct * 100, 4),
            'amount':        inv_call,
            'due_date':      (month + timedelta(days=10)).strftime('%Y-%m-%d'),
            'status':        'Settled' if m_idx < 10 else 'Pending',
        })

        # Distribution notice
        inv_dist = round(dist_amt * pct * 1_000_000, 2)
        dist_records.append({
            'date':          month.strftime('%Y-%m-%d'),
            'investor_id':   inv['investor_id'],
            'investor_name': inv['investor_name'],
            'jurisdiction':  inv['jurisdiction'],
            'dist_type':     np.random.choice(['Return of Capital','Realized Gain','Income'], p=[0.4,0.4,0.2]),
            'fund_total_m':  dist_amt,
            'investor_pct':  round(pct * 100, 4),
            'amount':        inv_dist,
            'payment_date':  (month + timedelta(days=15)).strftime('%Y-%m-%d'),
            'status':        'Paid' if m_idx < 10 else 'Pending',
        })

calls_df = pd.DataFrame(call_records)
dists_df = pd.DataFrame(dist_records)

total_called = call_amounts.sum()
total_dist   = dist_amounts.sum()

print(f'Capital call records: {len(calls_df):,}')
print(f'Distribution records: {len(dists_df):,}')
print(f'Total called (12 months): ${total_called:.1f}M  ({total_called/total_comm*100:.1f}% of commitments)')
print(f'Total distributed: ${total_dist:.1f}M')
print(f'\nMonthly call summary:')
for i,(m,c,d) in enumerate(zip(MONTHS,call_amounts,dist_amounts)):
    print(f'  {m.strftime("%b %Y")}: Call ${c}M  Dist ${d}M')

Capital call records: 468
Distribution records: 468
Total called (12 months): $230.6M  (41.9% of commitments)
Total distributed: $124.3M

Monthly call summary:
  Jan 2024: Call $30.7M  Dist $18.1M
  Feb 2024: Call $23.0M  Dist $13.6M
  Mar 2024: Call $26.1M  Dist $8.3M
  Apr 2024: Call $8.9M  Dist $8.1M
  May 2024: Call $7.4M  Dist $15.7M
  Jun 2024: Call $27.1M  Dist $10.6M
  Jul 2024: Call $14.1M  Dist $4.0M
  Aug 2024: Call $20.1M  Dist $16.4M
  Sep 2024: Call $11.7M  Dist $11.1M
  Oct 2024: Call $20.3M  Dist $6.2M
  Nov 2024: Call $16.2M  Dist $3.9M
  Dec 2024: Call $25.0M  Dist $8.3M


In [6]:
# Step 6 — Reporting Schedule Generation
# Monthly, quarterly, and annual reports per investor per fund type

np.random.seed(99)
report_records = []

for month_dt in MONTHS:
    month_str = month_dt.strftime('%Y-%m')
    month_num = month_dt.month
    is_quarter_end = month_num in [3, 6, 9, 12]
    is_year_end    = month_num == 12

    for _, inv in inv_df[inv_df['kyc_status'] == 'Approved'].iterrows():
        freq = REPORTING_FREQ[inv['fund_type']]

        # Monthly reports — Hedge Fund only
        if freq == 'Monthly':
            report_records.append({
                'period':        month_str,
                'investor_id':   inv['investor_id'],
                'investor_name': inv['investor_name'],
                'fund_type':     inv['fund_type'],
                'report_type':   'NAV Statement',
                'frequency':     'Monthly',
                'due_date':      f'{month_str}-15',
                'side_letter':   inv['side_letter'],
                'sl_obligation': 'Monthly NAV' if inv['side_letter'] else None,
                'status':        'Distributed' if month_num < 11 else 'Pending',
            })

        # Quarterly reports
        if is_quarter_end:
            for rtype in ['Capital Account Statement', 'Quarterly Report']:
                report_records.append({
                    'period':        month_str,
                    'investor_id':   inv['investor_id'],
                    'investor_name': inv['investor_name'],
                    'fund_type':     inv['fund_type'],
                    'report_type':   rtype,
                    'frequency':     'Quarterly',
                    'due_date':      f'{month_str}-30',
                    'side_letter':   inv['side_letter'],
                    'sl_obligation': 'Quarterly ESG Report' if (inv['side_letter'] and rtype == 'Quarterly Report') else None,
                    'status':        'Distributed' if month_num < 10 else 'Pending',
                })

        # Annual reports
        if is_year_end:
            for rtype in ['Annual Report', 'ILPA Report']:
                report_records.append({
                    'period':        month_str,
                    'investor_id':   inv['investor_id'],
                    'investor_name': inv['investor_name'],
                    'fund_type':     inv['fund_type'],
                    'report_type':   rtype,
                    'frequency':     'Annual',
                    'due_date':      f'{month_str}-31',
                    'side_letter':   inv['side_letter'],
                    'sl_obligation': 'ILPA Reporting' if (inv['side_letter'] and rtype == 'ILPA Report') else None,
                    'status':        'Pending',
                })

rpt_df = pd.DataFrame(report_records)
sl_rpts = rpt_df[rpt_df['side_letter'] == True]
missed  = rpt_df[rpt_df['status'] == 'Pending']

print(f'Total report obligations: {len(rpt_df):,}')
print(f'Reports distributed: {len(rpt_df[rpt_df.status=="Distributed"]):,}')
print(f'Reports pending: {len(missed):,}')
print(f'Side letter reports: {len(sl_rpts):,}')
print(f'\nBreakdown by frequency:')
print(rpt_df.groupby("frequency")["investor_id"].count().to_string())
print(f'\nBreakdown by report type:')
print(rpt_df.groupby("report_type")["investor_id"].count().to_string())

Total report obligations: 510
Reports distributed: 334
Reports pending: 176
Side letter reports: 272

Breakdown by frequency:
frequency
Annual        78
Monthly      120
Quarterly    312

Breakdown by report type:
report_type
Annual Report                 39
Capital Account Statement    156
ILPA Report                   39
NAV Statement                120
Quarterly Report             156


In [7]:
# Step 7 — Capital Account Statements
# Investor-level capital account position at fund year-end

np.random.seed(111)
cap_accounts = []

for _, inv in inv_df[inv_df['kyc_status'] == 'Approved'].iterrows():
    called     = inv['capital_called_m']
    uncalled   = inv['uncalled_comm_m']
    gross_mult = np.random.uniform(1.3, 2.2)
    gross_val  = round(called * gross_mult, 2)
    net_val    = round(gross_val * np.random.uniform(0.88, 0.96), 2)
    realized   = round(called * np.random.uniform(0.1, 0.5), 2)
    unrealized = round(net_val - realized, 2)
    mgmt_fee   = round(inv['commitment_m'] * 0.02, 3)
    carry      = round(max(0, (net_val - called) * 0.20), 2)
    nav        = round(net_val - carry, 2)

    cap_accounts.append({
        'investor_id':      inv['investor_id'],
        'investor_name':    inv['investor_name'],
        'jurisdiction':     inv['jurisdiction'],
        'fund_type':        inv['fund_type'],
        'commitment_m':     inv['commitment_m'],
        'capital_called_m': called,
        'uncalled_comm_m':  uncalled,
        'gross_value_m':    gross_val,
        'net_value_m':      net_val,
        'realized_m':       realized,
        'unrealized_m':     unrealized,
        'mgmt_fee_m':       mgmt_fee,
        'carried_interest': carry,
        'nav_m':            nav,
        'gross_multiple':   round(gross_val / called, 2) if called > 0 else 0,
        'net_multiple':     round(nav / called, 2) if called > 0 else 0,
        'irr_estimate_pct': round(np.random.uniform(8, 22), 1),
    })

cap_df = pd.DataFrame(cap_accounts)

print('CAPITAL ACCOUNT SUMMARY (Fund Year-End 2024)')
print('='*60)
print(f'  Approved investors:     {len(cap_df)}')
print(f'  Total commitments:      ${cap_df.commitment_m.sum():.1f}M')
print(f'  Total called:           ${cap_df.capital_called_m.sum():.1f}M')
print(f'  Total NAV:              ${cap_df.nav_m.sum():.1f}M')
print(f'  Total realized:         ${cap_df.realized_m.sum():.1f}M')
print(f'  Gross MOIC (avg):       {cap_df.gross_multiple.mean():.2f}x')
print(f'  Net MOIC (avg):         {cap_df.net_multiple.mean():.2f}x')
print(f'  Avg IRR estimate:       {cap_df.irr_estimate_pct.mean():.1f}%')

CAPITAL ACCOUNT SUMMARY (Fund Year-End 2024)
  Approved investors:     39
  Total commitments:      $433.1M
  Total called:           $214.8M
  Total NAV:              $317.7M
  Total realized:         $58.8M
  Gross MOIC (avg):       1.72x
  Net MOIC (avg):         1.47x
  Avg IRR estimate:       15.6%


In [8]:
# Step 8 — Fund Liquidation Checklist

LIQUIDATION_CHECKLIST = [
    {'item': 'Final NAV calculation confirmed by administrator',          'owner': 'Administrator',      'status': 'Complete'},
    {'item': 'All outstanding capital calls settled',                    'owner': 'Capital Markets',    'status': 'Complete'},
    {'item': 'Management fee calculations finalised and agreed',         'owner': 'Finance',            'status': 'Complete'},
    {'item': 'Carried interest waterfall calculated and approved',       'owner': 'Finance / GP',       'status': 'Complete'},
    {'item': 'Legal counsel sign-off on distribution mechanics',         'owner': 'Legal',              'status': 'Complete'},
    {'item': 'MLRO final AML check on all receiving accounts',          'owner': 'Compliance',         'status': 'Complete'},
    {'item': 'Tax withholding calculations per jurisdiction',            'owner': 'Tax',                'status': 'Complete'},
    {'item': 'Side letter obligations reviewed for liquidation clauses', 'owner': 'Client Services',   'status': 'Complete'},
    {'item': 'Investor bank account details verified',                   'owner': 'Client Services',   'status': 'Complete'},
    {'item': 'Transfer agent instruction templates prepared',            'owner': 'Transfer Agent',     'status': 'Complete'},
    {'item': 'Regulatory filings completed (AIFMD, Form PF)',           'owner': 'Compliance',         'status': 'Complete'},
    {'item': 'Fund board resolution approving liquidation',              'owner': 'Board',              'status': 'Complete'},
    {'item': 'Final audit sign-off from external auditors',             'owner': 'Deloitte',           'status': 'Pending'},
    {'item': 'Distribution notices drafted and approved by legal',       'owner': 'Legal',              'status': 'Pending'},
    {'item': 'FATCA/CRS reporting finalised for final year',            'owner': 'Compliance',         'status': 'Pending'},
    {'item': 'Side letter investors notified of liquidation timeline',   'owner': 'Client Services',   'status': 'Pending'},
    {'item': 'Final capital account statements issued to all LPs',       'owner': 'Client Services',   'status': 'Pending'},
    {'item': 'Liquidation payments released via transfer agent',         'owner': 'Transfer Agent',     'status': 'Pending'},
    {'item': 'Deregister fund with regulatory bodies (FCA/SEC)',        'owner': 'Compliance',         'status': 'Post-Distribution'},
    {'item': 'Archive all fund records per 7-year retention policy',    'owner': 'Operations',         'status': 'Post-Distribution'},
    {'item': 'Close all fund bank accounts',                             'owner': 'Finance',            'status': 'Post-Distribution'},
    {'item': 'Final GP report issued to all investors',                  'owner': 'Client Services',   'status': 'Post-Distribution'},
    {'item': 'Wind down operational infrastructure',                     'owner': 'IT / Operations',   'status': 'Post-Distribution'},
    {'item': 'Fund formally dissolved',                                  'owner': 'Legal',              'status': 'Post-Distribution'},
]

liq_df = pd.DataFrame(LIQUIDATION_CHECKLIST)
complete = len(liq_df[liq_df.status=='Complete'])
pending  = len(liq_df[liq_df.status=='Pending'])
post     = len(liq_df[liq_df.status=='Post-Distribution'])

print('FUND LIQUIDATION CHECKLIST STATUS')
print('='*55)
print(f'  Complete:          {complete} items')
print(f'  Pending:           {pending} items  <-- blocking distribution')
print(f'  Post-distribution: {post} items')
print(f'  Overall progress:  {complete}/{len(liq_df)} ({complete/len(liq_df)*100:.0f}%)')
print()
print('PENDING ITEMS (must resolve before release):')
for _, row in liq_df[liq_df.status=='Pending'].iterrows():
    print(f'  [{row.owner}] {row.item}')

FUND LIQUIDATION CHECKLIST STATUS
  Complete:          12 items
  Pending:           6 items  <-- blocking distribution
  Post-distribution: 6 items
  Overall progress:  12/24 (50%)

PENDING ITEMS (must resolve before release):
  [Deloitte] <bound method IndexOpsMixin.item of item      Final audit sign-off from external auditors
owner                                        Deloitte
status                                        Pending
Name: 12, dtype: object>
  [Legal] <bound method IndexOpsMixin.item of item      Distribution notices drafted and approved by l...
owner                                                 Legal
status                                              Pending
Name: 13, dtype: object>
  [Compliance] <bound method IndexOpsMixin.item of item      FATCA/CRS reporting finalised for final year
owner                                       Compliance
status                                         Pending
Name: 14, dtype: object>
  [Client Services] <bound method IndexOpsMi

In [9]:
# Step 9 — Excel Audit Export

with pd.ExcelWriter('alternatives_fund_lifecycle_2024.xlsx', engine='openpyxl') as writer:

    # Sheet 1: Investor Register
    inv_export = inv_df[[
        'investor_id','investor_name','investor_type','jurisdiction','fund_type',
        'commitment_m','kyc_status','risk_rating','fatca_crs','sub_doc_status',
        'side_letter','sl_obligations','mlro_signoff','capital_called_m',
        'uncalled_comm_m','reporting_freq','onboarding_date'
    ]].copy()
    inv_export.to_excel(writer, sheet_name='Investor Register', index=False)

    # Sheet 2: AML/KYC Status Report
    kyc_report = inv_df[['investor_id','investor_name','jurisdiction','kyc_status',
                          'risk_rating','fatca_crs','mlro_signoff','sub_doc_status','onboarding_date']].copy()
    kyc_report.to_excel(writer, sheet_name='AML-KYC Status', index=False)

    # Sheet 3: Capital Calls
    calls_df[['date','investor_id','investor_name','jurisdiction','purpose',
              'fund_total_m','investor_pct','amount','due_date','status']].to_excel(
        writer, sheet_name='Capital Calls', index=False)

    # Sheet 4: Distribution Notices
    dists_df[['date','investor_id','investor_name','jurisdiction','dist_type',
              'fund_total_m','investor_pct','amount','payment_date','status']].to_excel(
        writer, sheet_name='Distribution Notices', index=False)

    # Sheet 5: Capital Account Statements
    cap_df.to_excel(writer, sheet_name='Capital Accounts', index=False)

    # Sheet 6: Reporting Schedule
    rpt_df[['period','investor_id','investor_name','fund_type','report_type',
            'frequency','due_date','side_letter','sl_obligation','status']].to_excel(
        writer, sheet_name='Reporting Schedule', index=False)

    # Sheet 7: Side Letter Register
    sl_df = inv_df[inv_df.side_letter == True][[
        'investor_id','investor_name','jurisdiction','fund_type','commitment_m',
        'sl_obligations','reporting_freq','kyc_status'
    ]].copy()
    sl_df.to_excel(writer, sheet_name='Side Letter Register', index=False)

    # Sheet 8: Liquidation Checklist
    liq_df.to_excel(writer, sheet_name='Liquidation Checklist', index=False)

print('Export complete: alternatives_fund_lifecycle_2024.xlsx')
print('  Sheet 1: Investor Register (50 investors)')
print('  Sheet 2: AML/KYC Status Report')
print('  Sheet 3: Capital Calls (12 months)')
print('  Sheet 4: Distribution Notices (12 months)')
print('  Sheet 5: Capital Account Statements')
print('  Sheet 6: Reporting Schedule')
print('  Sheet 7: Side Letter Register')
print('  Sheet 8: Liquidation Checklist')

Export complete: alternatives_fund_lifecycle_2024.xlsx
  Sheet 1: Investor Register (50 investors)
  Sheet 2: AML/KYC Status Report
  Sheet 3: Capital Calls (12 months)
  Sheet 4: Distribution Notices (12 months)
  Sheet 5: Capital Account Statements
  Sheet 6: Reporting Schedule
  Sheet 7: Side Letter Register
  Sheet 8: Liquidation Checklist


In [10]:
# Step 10 — Final Summary

total_comm_all = inv_df['commitment_m'].sum()
total_called   = calls_df.groupby('date')['fund_total_m'].first().sum()
total_dist     = dists_df.groupby('date')['fund_total_m'].first().sum()

print('PROJECT COMPLETE — ALTERNATIVES FUND INVESTOR LIFECYCLE TRACKER')
print('='*65)
print(f'  Total investors:              {len(inv_df)}')
print(f'  Jurisdictions:                {inv_df.jurisdiction.nunique()}')
print(f'  Fund strategies:              {inv_df.fund_type.nunique()}')
print(f'  Total commitments:            ${total_comm_all:.1f}M')
print(f'  KYC Approved:                 {len(inv_df[inv_df.kyc_status=="Approved"])} ({len(inv_df[inv_df.kyc_status=="Approved"])/len(inv_df)*100:.0f}%)')
print(f'  KYC Pending:                  {len(inv_df[inv_df.kyc_status=="Pending"])}')
print(f'  KYC Escalated:                {len(inv_df[inv_df.kyc_status=="Escalated"])}')
print(f'  Side letter investors:        {inv_df.side_letter.sum()}')
print(f'  Capital calls (12 months):    ${total_called:.1f}M')
print(f'  Distributions (12 months):    ${total_dist:.1f}M')
print(f'  Total report obligations:     {len(rpt_df):,}')
print(f'  Liquidation checklist:        {complete}/{len(liq_df)} complete')
print(f'  Excel sheets exported:        8')
print()
print('Files: alternatives_fund_lifecycle_2024.xlsx')

PROJECT COMPLETE — ALTERNATIVES FUND INVESTOR LIFECYCLE TRACKER
  Total investors:              50
  Jurisdictions:                8
  Fund strategies:              4
  Total commitments:            $550.8M
  KYC Approved:                 39 (78%)
  KYC Pending:                  5
  KYC Escalated:                6
  Side letter investors:        20
  Capital calls (12 months):    $230.6M
  Distributions (12 months):    $124.3M
  Total report obligations:     510
  Liquidation checklist:        12/24 complete
  Excel sheets exported:        8

Files: alternatives_fund_lifecycle_2024.xlsx
